## Setup Unsloth for 4-bit Quantization

First, we need to install the `unsloth` library and its dependencies. This process includes installing `xformers` and `bitsandbytes` for efficient 4-bit quantization.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Put this at the very top of your notebook, before any imports

In [2]:
# Install bitsandbytes separately from PyPI
!pip install bitsandbytes
# Install Unsloth and fixed versions of dependencies to avoid building from source
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.30" "trl<0.9.0" peft accelerate bitsandbytes
!pip install unsloth_zoo
!pip install xgrammar


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 53.4 MB/s eta 0:00:00
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-pgbeusxr/unsloth_5c6c833dd48e48dab262cf9b60c25c1c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-pgbeusxr/unsloth_5c6c833dd48e48dab262cf9b60c25c1c
  Resolved https://github.com/unslothai/unsloth.git to commit 85314ed1622cbdcbd6621ebf97307606f4baf89d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.6.1-py3-none-any.whl size=34766650 sha256=8f890220de661d81fbca650fe351cbc7325940af8c9a02ae3676b24a6bfbe9e2
  Stored in directory: /tmp/pip-ephem-wheel-cache-fxmb3ivm/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 68.0 MB/s eta 0:00:00
   ━━━━━━

In [7]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-9B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.6.1: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/876 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

### 6. Run Native TritonBench
Let's clone the official TritonBench repository and generate our own `predictions.jsonl` directly using the fine-tuned Unsloth model.

In [12]:
%cd /content
!git clone https://github.com/thunlp/TritonBench.git
%cd TritonBench
!ls -la data/

/content
fatal: destination path 'TritonBench' already exists and is not an empty directory.
/content/TritonBench
total 59844
drwxr-xr-x 4 root root     4096 Jun  9 07:57 .
drwxr-xr-x 8 root root     4096 Jun  9 08:57 ..
-rw-r--r-- 1 root root 24569203 Jun  9 07:57 train_crawl.json
-rw-r--r-- 1 root root 32123591 Jun  9 07:57 train_synth.json
-rw-r--r-- 1 root root  1245074 Jun  9 07:57 TritonBench_G_comp_alpac_v1.json
-rw-r--r-- 1 root root  1116964 Jun  9 07:57 TritonBench_G_simp_alpac_v1.json
drwxr-xr-x 2 root root    12288 Jun  9 07:57 TritonBench_G_v1
-rw-r--r-- 1 root root  1423105 Jun  9 07:57 TritonBench_G_v1.json
-rw-r--r-- 1 root root   253412 Jun  9 07:57 TritonBench_T_comp_alpac_v1.json
-rw-r--r-- 1 root root   202822 Jun  9 07:57 TritonBench_T_simp_alpac_v1.json
drwxr-xr-x 2 root root    12288 Jun  9 07:57 TritonBench_T_v1
-rw-r--r-- 1 root root   299814 Jun  9 07:57 TritonBench_T_v1.jsonl


In [ ]:
import json
import os
from tqdm import tqdm
import xgrammar as xgr
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, LogitsProcessorList

# 1. Define grammar
GRAMMAR = r"""
root ::= imports newline* func-def newline*

imports ::= "import triton" newline "import triton.language as tl" newline

newline ::= "\n" | "\r\n"
ws      ::= [ \t]*
wsnl    ::= [ \n\t\r]*
i1      ::= "    " | "\t"
i2      ::= "        " | "\t\t"
i3      ::= "            " | "\t\t\t"

name   ::= [a-zA-Z_][a-zA-Z0-9_]*
number ::= [0-9]+ ("." [0-9]*)? ([eE][+-]?[0-9]+)?
string ::= "\"" [^"]* "\"" | "'" [^']* "'"

func-def ::= "@triton.jit" newline "def" ws name ws "(" ws paramlist ws ")" ws ":" newline block1

paramlist ::= (param (ws "," ws param)* (ws ",")?)?
param ::= name ws ":" ws "tl.constexpr"
        | name ws "=" ws expr
        | name

block1 ::= (i1 stmt newline | i1 if-stmt | i1 for-stmt)+
block2 ::= (i2 stmt newline | i2 if-stmt2 | i2 for-stmt2)+
block3 ::= (i3 stmt newline)+

stmt ::= assign
       | store
       | return-stmt
       | expr

assign ::= target ws "=" ws expr
target ::= name | name "[" expr "]" | name "." name

return-stmt ::= "return" ws expr | "return"

if-stmt ::= "if" ws expr ws ":" newline block2
if-stmt2 ::= "if" ws expr ws ":" newline block3

for-stmt ::= "for" ws name ws "in" ws expr ws ":" newline block2
for-stmt2 ::= "for" ws name ws "in" ws expr ws ":" newline block3

store ::= "tl.store" ws "(" ws args ws ")"

expr ::= comparison
comparison ::= arith (ws cmp-op ws arith)*
cmp-op ::= "<=" | ">=" | "==" | "!=" | "<" | ">"

arith ::= term (ws add-op ws term)*
add-op ::= "+" | "-" | "|" | "&" | "^"

term ::= factor (ws mul-op ws factor)*
mul-op ::= "*" | "/" | "//" | "%" | "@"

factor ::= unary | power
unary ::= ("+" | "-" | "~") ws factor
power ::= atom (ws "**" ws factor)?

atom ::= number
       | string
       | "True"
       | "False"
       | "None"
       | call
       | attr
       | index
       | tuple
       | name
       | "(" ws expr ws ")"

attr ::= name "." name
index ::= name "[" index-expr "]"

tuple ::= "(" ws expr ws "," ws expr (ws "," ws expr)* (ws ",")? ws ")"

call ::= callable ws "(" ws args? ws ")"
callable ::= name | name "." name | name "." name "." name

args ::= arg (ws "," ws arg)* (ws ",")?
arg ::= name ws "=" ws expr
      | expr

index-expr ::= expr
             | expr? ws ":" ws expr?
             | expr? ws ":" ws expr? ws ":" ws expr?
"""

# 2. Compile grammar with XGrammar

if hasattr(tokenizer, "tokenizer"):
    processor = tokenizer
    tokenizer = processor.tokenizer

full_vocab_size = getattr(model.config, "vocab_size", None)

if full_vocab_size is None:
    full_vocab_size = len(tokenizer)

tokenizer_info = xgr.TokenizerInfo.from_huggingface(
    tokenizer,
    vocab_size=full_vocab_size
)

compiler = xgr.GrammarCompiler(tokenizer_info)

compiled_grammar = compiler.compile_grammar(
    xgr.Grammar.from_ebnf(GRAMMAR)
)

data_path = "/content/TritonBench/data/TritonBench_T_simp_alpac_v1.json"
output_file = "/content/predictions.jsonl"

BATCH_SIZE = 1

MAX_NEW_TOKENS = 512

if os.path.exists(data_path):
    with open(data_path, "r") as f:
        eval_data = json.load(f)

    print(f"Loaded {len(eval_data)} benchmark items.")

    tokenizer.padding_side = "left"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    results = []

    for i in tqdm(range(0, len(eval_data), BATCH_SIZE)):
        batch = eval_data[i : i + BATCH_SIZE]

        batch_prompts = []

        for item in batch:
            messages = [
                {
                    "role": "system",
                    "content": """You are an expert GPU programmer specialized in writing Triton kernels.

Given a PyTorch function, write an equivalent Triton implementation.

Return only executable Python code.
Do not use Markdown.
Do not explain anything.
Do not include natural language.

Correctness is more important than performance.
Compilation is more important than optimization.

Requirements:
- Use import triton
- Use import triton.language as tl
- Use @triton.jit
- Use tl.program_id
- Use tl.arange
- Use tl.load and tl.store
- Use masks for boundary accesses
- Use tl.constexpr for compile-time constants
- Do not use torch inside @triton.jit
- Do not use numpy inside @triton.jit
- Do not use Python len() on tensors inside kernels
- Preserve the same output shape and numerical behavior as the PyTorch function
"""
                },
                {
                    "role": "user",
                    "content": item["instruction"]
                },
            ]

            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )

            batch_prompts.append(prompt)

        encoded = tokenizer(
            text=batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048,
            add_special_tokens=False,
        ).to("cuda")

        xgr_logits_processor = xgr.contrib.hf.LogitsProcessor(compiled_grammar)

        import time
        start = time.time()

        outputs = model.generate(
            input_ids=encoded["input_ids"],
            attention_mask=encoded["attention_mask"],
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
            logits_processor=LogitsProcessorList([xgr_logits_processor]),
        )

        print("Generation time:", time.time() - start)

        for j, (item, output) in enumerate(zip(batch, outputs)):
            input_len = encoded["attention_mask"][j].sum().item()
            generated_tokens = output[input_len:]

            triton_code = tokenizer.decode(
                generated_tokens,
                skip_special_tokens=True
            ).strip()

            results.append({
                "instruction": item["instruction"],
                "predict": triton_code
            })

    with open(output_file, "w") as out_f:
        for res in results:
            out_f.write(json.dumps(res) + "\n")

    print(f"\nSaved {len(results)} predictions to {output_file}")

else:
    print(f"Data file not found: {data_path}")

Loaded 166 benchmark items.


  1%|          | 1/166 [01:58<5:27:09, 118.97s/it]

Generation time: 118.958167552948


  1%|          | 2/166 [03:22<4:28:47, 98.34s/it] 

Generation time: 83.89411401748657


  2%|▏         | 3/166 [04:42<4:03:35, 89.66s/it]

Generation time: 79.33558416366577


In [ ]:
import json
import ast
from tqdm import tqdm

predictions_path = "/content/predictions.json"

stats = {
    "total": 0,
    "syntax_ok": 0,
    "syntax_fail": 0,
    "empty": 0,
    "contains_triton": 0,
}

syntax_errors = []

with open(predictions_path, "r") as f:
    predictions = [json.loads(line) for line in f]

for idx, item in enumerate(tqdm(predictions)):
    stats["total"] += 1

    code = item["predict"].strip()

    if not code:
        stats["empty"] += 1
        continue

    if "triton" in code.lower():
        stats["contains_triton"] += 1

    try:
        ast.parse(code)
        stats["syntax_ok"] += 1
    except Exception as e:
        stats["syntax_fail"] += 1
        syntax_errors.append(
            {
                "idx": idx,
                "error": str(e)
            }
        )

print("\n===== PRECHECK =====")
for k, v in stats.items():
    print(f"{k:20} {v}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/predictions.jsonl'

### 7. Clean and Prepare Kernels for TritonBench
We need to extract the actual Python code from the generated text and save them as individual `.py` files. We will use a regex to find python code blocks if they exist, or just try to clean the raw text.

In [ ]:
import json
import os
import re
import ast

predictions_path = "/content/predictions.json"
output_dir = "/content/eval_llm_outputs/valid"
os.makedirs(output_dir, exist_ok=True)

with open("/content/TritonBench/data/TritonBench_T_simp_alpac_v1.json", "r") as f:
    gold_data = json.load(f)

with open(predictions_path, "r") as f:
    predictions = [json.loads(line) for line in f]

cleaned_stats = {"total": 0, "syntax_ok": 0, "syntax_fail": 0}

for i, pred in enumerate(predictions):
    raw_text = pred["predict"]

    text_no_think = re.sub(r"<think>.*?</think>", "", raw_text, flags=re.DOTALL)

    match = re.search(r"```python\n(.*?)```", text_no_think, flags=re.DOTALL)
    if match:
        code = match.group(1).strip()
    else:
        code = text_no_think.strip()
        code = re.sub(r"^=[0-9]+\s*", "", code)

    cleaned_stats["total"] += 1

    try:
        ast.parse(code)
        cleaned_stats["syntax_ok"] += 1

        # Only save if syntax is valid
        file_path = os.path.join(output_dir, f"{i}.py")
        with open(file_path, "w") as f:
            f.write(code)

    except SyntaxError:
        cleaned_stats["syntax_fail"] += 1
        print(f"Skipping prediction {i} due to syntax error.")

print("=== CLEANING RESULTS ===")
print(f"Total parsed: {cleaned_stats['total']}")
print(f"Valid Python Syntax (saved): {cleaned_stats['syntax_ok']}")
print(f"Invalid Syntax (skipped): {cleaned_stats['syntax_fail']}")
print(f"Saved to {output_dir}")

### 8. Run TritonBench Execution Accuracy
Now we run the execution evaluation. The script `1_exe_acc.py` uses multiprocessing to run the models simultaneously across the GPUs provided. We pass `0` to use the primary GPU (or `0,1,2,3` if you have multiple).

In [ ]:
# We need to patch the script slightly because it hardcodes the python interpreter and gold_folder path
import os

eval_script = "/content/TritonBench/EVAL/eval_T/1_exe_acc.py"
with open(eval_script, "r") as f:
    script_code = f.read()

# Patch interpreter to the colab environment python
script_code = script_code.replace('py_interpreter = "/home/lijianling/miniconda3/envs/LLM/bin/python"', 'py_interpreter = "python"')
# Patch gold folder to the simplified benchmark folder
script_code = script_code.replace('gold_folder = "data/TritonBench_T_v1/"', 'gold_folder = "/content/TritonBench/data/TritonBench_T_v1/"')

with open(eval_script, "w") as f:
    f.write(script_code)

print("Patched evaluation script.")

In [ ]:
!cd /content/TritonBench && python EVAL/eval_T/1_exe_acc.py --folder /content/eval_llm_outputs/valid --GPUs 0